# 10. Efficiency results

Display the latency measurements and encoder profiles as tables directly in this notebook.


## Measurement inputs

The tables below read saved measurements; this notebook does not run a GPU benchmark. Generate the latency measurements with `scripts/run_topiocqa_full_dev_latency.py` and the encoder profiles with `scripts/profile_topiocqa_encoder_nsight.py`, following the commands and prerequisites in the repository README. Keep the IterCQR checkpoint configuration at the documented model path. Rerunning the display uses your local results and stops if required inputs are missing.

### Model-processing time

The thesis measures $I_{512}$, $I_{64}$, $R_{64}$, $D_{64}$, and $IRD_{64}$ on all **2,514 TopiOCQA development queries**. The 205 queries without history bypass the selector. Batch size is 16, with dynamic padding, five warm-up batches per component, and five independent measured processes. The 95% Student-$t$ intervals use the five process means, with four degrees of freedom. Route times include selection and rewriting but exclude retrieval and rank fusion.

The notation $C_{R_{64}}$, for example, denotes mean model-processing time per query. On the thesis RTX 4070 Laptop GPU, $C_{I_{512}}=52.8\,\mathrm{ms}$ and $C_{R_{64}}=34.4\,\mathrm{ms}$, a 34.8% reduction. The paired difference is $-18.4\,\mathrm{ms}$ with a 95% interval of $[-27.8,-9.0]\,\mathrm{ms}$. These are reference values from the thesis, not a guarantee for another GPU or software environment.

### Encoder work and memory traffic

For one sequence of length $n$, the thesis estimates encoder work as

$$
F_{\mathrm{enc}}(n)\approx L_{\mathrm{enc}}\left(8nd_{\mathrm{model}}^2+4n^2d_{\mathrm{model}}+4nd_{\mathrm{model}}d_{\mathrm{ff}}\right).
$$

Here $d_{\mathrm{model}}=768$, $d_{\mathrm{ff}}=3072$, and $L_{\mathrm{enc}}=12$. For the uniform synthetic batches of 16, the thesis uses $n=150$ for $I_{512}$ and $n=41$ for $R_{64}$. Thus $16F_{\mathrm{enc}}(150)\approx421.0\,\mathrm{GFLOP}$ and $16F_{\mathrm{enc}}(41)\approx112.4\,\mathrm{GFLOP}$.

The measured FP32 proxy is a separate quantity,

$$
F_{\mathrm{FP32}}=N_{\mathrm{add}}+N_{\mathrm{mul}}+2N_{\mathrm{fma}},
$$

where the counts refer to per-thread additions, multiplications, and fused multiply-adds. The thesis reports 458.9 and 127.0 GFLOP per batch, respectively. Range replay measures full-pass DRAM traffic of 5.33 and 1.38 GB; component traffic is an estimate based on kernel-replay shares. These byte transfers are not the storage estimate $M_{\mathrm{enc}}(n)$, and encoder profiles are not complete route timings.


In [ ]:
from pathlib import Path
import os
import sys

configured_root = os.environ.get("ROCC_ROOT") or os.environ.get("MASTER_THESIS_PROJECT_ROOT")
if configured_root:
    PROJECT_ROOT = Path(configured_root).expanduser().resolve()
else:
    PROJECT_ROOT = next(
        (path for path in (Path.cwd(), *Path.cwd().parents)
         if (path / "experiments" / "rocc").is_dir()),
        None,
    )
if PROJECT_ROOT is None or not (PROJECT_ROOT / "experiments" / "rocc").is_dir():
    raise FileNotFoundError("Open this notebook from the ROCC checkout or set ROCC_ROOT to it.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [10]:
from experiments.scripts.analysis_results import display_results

display_results(PROJECT_ROOT / "experiments", '10_efficiency_analysis')


**measurement setup**

,independent_runs,queries,selector_queries,zero_history_queries,batch_size,warmup_batches,shared_I_R_inputs,unique_IRD_rewriter_inputs
0,5,2514,2309,205,16,5,212,4816


**model processing ms per query**

,route,budget,selection_ms,encoder_ms,decoder_ms,total_ms,ci95_low_ms,ci95_high_ms
0,I,64,0.000000,1.890684,26.833103,28.723788,20.356867,37.090708
1,I,512,0.000000,11.098205,41.670269,52.768474,38.926703,66.610246
2,R,64,7.870067,1.787045,24.722506,34.379618,27.177814,41.581422
3,D,64,7.870067,0.000000,0.000000,7.870067,5.628065,10.112069
4,IRD,64,7.870067,3.449811,45.689356,57.009234,49.994103,64.024365


**paired time differences vs I512**

...

,route,budget,component,GFLOP,estimated_range_DRAM_GB,kind
0,I,64,ffn,109.273743,0.799547,measured encoder work; estimated DRAM allocation
1,I,64,other,0.109575,0.486922,measured encoder work; estimated DRAM allocation
2,I,64,self_attention,64.451950,0.861320,measured encoder work; estimated DRAM allocation
3,I,512,ffn,275.728564,1.495095,measured encoder work; estimated DRAM allocation
4,I,512,other,0.288376,1.199024,measured encoder work; estimated DRAM allocation
5,I,512,self_attention,182.857305,2.636040,measured encoder work; estimated DRAM allocation
6,R,64,ffn,80.134078,0.543546,measured encoder work; estimated DRAM allocation
7,R,64,other,0.078833,0.300854,measured encoder work; estimated DRAM allocation
8,R,64,self_attention,46.815862,0.538515,measured encoder work; estimated DRAM allocation
9,IRD,64,ffn,189.407822,1.343093,sum of I64 and R64 proxies


**analytical encoder work**

,route,budget,n,batch_size,d_model,d_ff,encoder_layers,analytic_GFLOP
0,I,512,150,16,768,3072,12,420.957389
1,R,64,41,16,768,3072,12,112.425763


**input length ratios**

,reference,selection,linear_length_ratio,quadratic_length_ratio
0,I512,R64,3.658537,13.38489
